# 14 · MCA y análisis factorial: perfiles de propiedades y estructura barrial

## Preguntas y continuidad
- ¿Las modalidades de atributos inmobiliarios forman configuraciones que también distinguen alquiler temporario, alquiler permanente y venta?
- ¿Los perfiles de servicios/transporte barriales pueden resumirse en factores útiles para interpretar KPIs?
- En notebook 03, los bloques de coausencia de nulos (cuadrados de variables presentes/ausentes) se interpretaron como posible cobertura/esquema de carga por fuente. No se tomarán como factores de mercado: nulos y fuente quedan fuera del espacio activo. “No informado” no se convierte en atributo.
- Conecta con PCA/MCA territorial (10), correspondencia de etiquetas y perfiles de propiedades (13), KPIs priorizados (09) y modelos explicativos (12). El dashboard es posterior.
- Asociaciones descriptivas no prueban preferencia, causalidad, sustitución ni desplazamiento (H4).

In [ ]:
# Nbconvert ejecuta desde notebooks/; buscar la raíz del repositorio hacia arriba.
ROOT=next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/"data"/"processed").exists()), Path.cwd())
DATA=ROOT/"data"/"processed"
def read(name):
    p=DATA/name
    if not p.exists(): raise FileNotFoundError(f"Falta {p}; ejecutar notebooks previos.")
    return pd.read_csv(p, low_memory=False)
listings=read("df_publicaciones_enriquecidas_v1.csv")
kpi=read("kpis_barrio_v1.csv")
print("Publicaciones",len(listings),"barrios KPI",len(kpi))

## MCA de propiedades

- Rasgos activos: tipología/amenities y categorías de superficie, ambientes, baños y contexto urbano, sujetos a cobertura ≥65%.
- Operación, fuente, barrio y precio quedan suplementarios o fuera del análisis; así se evalúa correspondencia sin circularidad.
- Cuantitativas se agrupan en terciles, dejando visibles cobertura, casos completos y pérdida muestral.
- MCA sintetiza asociaciones entre modalidades; complementar con V de Cramér y BH en las modalidades más frecuentes.

In [ ]:
cand_cat=[c for c in ["tipo_propiedad","amenities_mencionadas"] if c in listings]
cand_num=[c for c in ["superficie_ref_m2","ambientes_ref","dormitorios_ref","banios_ref",
 "dist_transporte_masivo_m","densidad_educacion_km2","densidad_gastronomia_km2",
 "pct_superficie_verde"] if c in listings]
op=next((c for c in ["tipo_operacion_norm","tipo_operacion"] if c in listings),None)
supp=[c for c in [op,"fuente_norm","barrio_oficial"] if c and c in listings]
active=cand_cat+cand_num
if len(active)<3: raise ValueError(f"Variables insuficientes: {active}")
coverage=listings[active].notna().mean().sort_values()
display(coverage.to_frame("cobertura"))
active=[c for c in active if coverage[c]>=.65]
if len(active)<3: raise ValueError(f"Menos de tres variables activas superan cobertura: {active}")
d=listings[active+supp].copy()
for c in cand_num:
    if c in active: d[c]=pd.qcut(d[c],3,duplicates="drop").astype("string")
for c in cand_cat:
    if c in active: d[c]=d[c].astype("string")
d=d.replace({"nan":np.nan,"None":np.nan}).dropna(subset=active)
n_before=len(listings)
if len(d)>50000:
    if op:
        parts=[]; total=len(d)
        for _,group in d.groupby(op,dropna=False):
            take=min(len(group),max(1,round(50000*len(group)/total)))
            parts.append(group.sample(n=take,random_state=42))
        d=pd.concat(parts).sample(frac=1,random_state=42)
    else:
        d=d.sample(50000,random_state=42)
print(f"Casos completos y analizados: {len(d):,}/{n_before:,} ({len(d)/n_before:.1%}); activas={active}")

In [ ]:
# MCA via SVD de residuos estandarizados de la matriz indicadora.
try:
    enc=OneHotEncoder(handle_unknown="ignore",sparse_output=True)
except TypeError:  # compatibilidad con scikit-learn <1.2
    enc=OneHotEncoder(handle_unknown="ignore",sparse=True)
Z=enc.fit_transform(d[active]).astype(float).toarray()
P=Z/Z.sum(); r=P.sum(1); c=P.sum(0)
S=(P-np.outer(r,c))/np.sqrt(np.outer(r,c))
U,s,Vt=svd(S,full_matrices=False)
eig=s**2; inertia=eig/eig.sum()
display(pd.DataFrame({"eigenvalue":eig[:5],"inercia_pct":100*inertia[:5],
 "acumulada_pct":100*np.cumsum(inertia[:5])},index=[f"Dim {i+1}" for i in range(min(5,len(eig)))]))
rows=pd.DataFrame(U[:,:2]*s[:2]/np.sqrt(r[:,None]),columns=["dim1","dim2"],index=d.index)
cats=pd.DataFrame(Vt[:2].T*s[:2]/np.sqrt(c[:,None]),columns=["dim1","dim2"])
cats["modalidad"]=enc.get_feature_names_out(active); cats["masa"]=c
fig,ax=plt.subplots(figsize=(11,8))
sns.scatterplot(data=cats,x="dim1",y="dim2",size="masa",sizes=(20,150),ax=ax)
for _,z in cats.iterrows(): ax.text(z.dim1,z.dim2,z.modalidad,fontsize=8)
ax.axhline(0,color="grey",lw=.6); ax.axvline(0,color="grey",lw=.6)
ax.set(title="MCA: modalidades activas de propiedades",xlabel="Dimensión 1",ylabel="Dimensión 2")
plt.tight_layout()

In [ ]:
# Etiquetas suplementarias: centroides, no utilizadas para estimar ejes.
for col in supp:
    cent=rows.join(d[col].rename(col)).groupby(col)[["dim1","dim2"]].mean()
    display(cent)
    ax=cent.plot.scatter("dim1","dim2",figsize=(7,5),title=f"Centroides suplementarios: {col}",grid=True)
    for label,z in cent.iterrows(): ax.annotate(str(label),(z.dim1,z.dim2))
# Asociación de operación con modalidades frecuentes; V de Cramér y corrección BH.
if op:
    Zdf=pd.DataFrame(Z,index=d.index,columns=enc.get_feature_names_out(active))
    tests=[]
    for m in cats.nlargest(10,"masa").modalidad:
        tab=pd.crosstab(d[op],Zdf[m]>0)
        if min(tab.shape)>1:
            stat,p,_,_=stats.chi2_contingency(tab)
            tests.append({"modalidad":m,"V_cramer":np.sqrt(stat/(tab.values.sum()*min(tab.shape)-tab.values.sum())),
                          "p":p,"n":tab.values.sum()})
    assoc=pd.DataFrame(tests)
    if len(assoc):
        from statsmodels.stats.multitest import multipletests
        assoc["q_BH"]=multipletests(assoc.p,method="fdr_bh")[1]
        display(assoc.sort_values("q_BH"))

### Interpretación y vínculo con hipótesis
- Operación suplementaria separada en el plano indica correspondencia descriptiva entre atributos publicados y modalidad; solapamiento sugiere que estos rasgos no distinguen bien las etiquetas.
- **H1:** MCA no estima concentración territorial; priorizar densidades/participaciones de oferta temporaria de 09–10.
- **H2a:** ambientes/dormitorios son capacidad aproximada, no composición de hogares; tamaño medio censal también es proxy indirecto.
- **H2b:** densidad de servicios expresa dotación, no acceso efectivo/calidad.
- **H3:** los ejes de transporte no prueban efecto sobre precio; mantener comparaciones por venta y alquiler separadas (12).
- **H4:** asociación transversal no identifica desplazamiento ni sustitución.
- Salvedades: complete-case puede sesgar cobertura; categorías por terciles dependen de empates; composición/cobertura de portales puede variar; se excluyen variables con cobertura <65%; V y p son asociativas y BH ajusta pruebas exploratorias.

## Análisis factorial exploratorio de indicadores barriales
Exploramos si las densidades de transporte/servicios y superficie verde pueden resumirse en dimensiones interpretables. El antecedente de coausencia del notebook 03 corresponde a patrón de captura y no aporta evidencia de factores latentes urbanos. Precios y KPIs de oferta se mantienen suplementarios.

In [ ]:
urban=[c for c in ["densidad_subte_km2","densidad_tren_km2","densidad_metrobus_km2",
 "densidad_hospitales_km2","densidad_educacion_km2","densidad_culturales_km2",
 "densidad_gastronomia_km2","densidad_espacios_verdes_km2","pct_superficie_verde"] if c in kpi]
supp_kpi=[c for c in ["airbnb_por_km2","mediana_precio_m2_venta_usd",
 "mediana_precio_m2_alquiler_usd","brecha_airbnb_vs_permanente_pct"] if c in kpi]
f=kpi.dropna(subset=urban).copy()
X=f[urban].replace([np.inf,-np.inf],np.nan).dropna(); f=f.loc[X.index]
Xz=StandardScaler().fit_transform(X); n,p=Xz.shape
R=np.corrcoef(Xz,rowvar=False); inv=np.linalg.pinv(R)
partial=-inv/np.sqrt(np.outer(np.diag(inv),np.diag(inv))); np.fill_diagonal(partial,0)
rr=R.copy(); np.fill_diagonal(rr,0); rr=rr**2
kmo=rr.sum()/(rr.sum()+(partial**2).sum())
bart=- (n-1-(2*p+5)/6)*np.log(max(np.linalg.det(R),1e-12))
bart_p=chi2.sf(bart,p*(p-1)/2)
obs=np.linalg.eigvalsh(R)[::-1]
rng=np.random.default_rng(42)
null=np.array([np.linalg.eigvalsh(np.corrcoef(rng.normal(size=(n,p)),rowvar=False))[::-1] for _ in range(500)])
threshold=np.quantile(null,.95,axis=0); nf=int((obs>threshold).sum())
print({"barrios":n,"variables":p,"KMO":round(kmo,3),"Bartlett_p":bart_p,"factores_parallel":nf})
display(pd.DataFrame({"autovalor":obs,"umbral_paralelo_95":threshold}))


In [ ]:
# No forzar factores: n pequeño, KMO/Bartlett/parallel analysis deben dar respaldo mínimo.
if n>=5*p and kmo>=.60 and bart_p<.05 and nf>=1:
    nf=min(nf,3)
    model=FactorAnalysis(n_components=nf,random_state=42).fit(Xz)
    L=model.components_.T
    def varimax(A,gamma=1,q=100,tol=1e-6):
        p_,k_=A.shape; rot=np.eye(k_); old=0
        for _ in range(q):
            B=A@rot
            u,s,v=np.linalg.svd(A.T@(B**3-(gamma/p_)*B@np.diag(np.diag(B.T@B))))
            rot=u@v; val=s.sum()
            if old and val/old<1+tol: break
            old=val
        return rot
    rot=varimax(L); load=pd.DataFrame(L@rot,index=urban,columns=[f"Factor {i+1}" for i in range(nf)])
    scores=pd.DataFrame(model.transform(Xz)@rot,index=f.index,columns=load.columns)
    display(load.round(3))
    sns.heatmap(load,annot=True,center=0,cmap="vlag",vmin=-1,vmax=1)
    plt.title("Cargas rotadas (Varimax)"); plt.tight_layout()
    from statsmodels.stats.multitest import multipletests
    rel=[]
    for fac in scores:
        for col in supp_kpi:
            pair=pd.concat([scores[fac],f[col]],axis=1).dropna()
            if len(pair)>=8:
                rho,pv=stats.spearmanr(pair.iloc[:,0],pair.iloc[:,1])
                rel.append({"factor":fac,"KPI":col,"n":len(pair),"rho":rho,"p":pv})
    rel=pd.DataFrame(rel)
    if len(rel):
        rel["q_BH"]=multipletests(rel.p,method="fdr_bh")[1]; display(rel.sort_values("q_BH"))
else:
    print("No se retiene solución factorial: adecuación, tamaño muestral o parallel analysis insuficientes.")

### Lectura condicionada y límites
- Solo nombrar factores luego de revisar cargas, comunalidades y estabilidad; no anticipar etiquetas.
- La relación con Airbnb/precios/brechas es Spearman exploratorio con ajuste BH; 48 barrios, múltiples comparaciones y autocorrelación espacial limitan inferencia.
- Variables de densidad comparten denominadores y pueden reflejar centralidad/tamaño urbano. KMO, Bartlett y parallel analysis no garantizan reproducibilidad.
- Sin estabilidad espacial/temporal y evidencia adicional, conservar factores como síntesis exploratoria, no como índice oficial de priorización.
- Para H1 priorizar oferta temporaria relativa; H2a requiere mejores datos de hogar; H2b y H3 describen asociaciones; H4 necesita snapshots temporales para evaluar cambios.

## Panel visual de síntesis: evidencia disponible y adecuación
Las tarjetas distinguen el alcance del MCA de la evidencia para análisis factorial. El panel inferior compara autovalores observados con el umbral de parallel analysis; solo los ejes que exceden el umbral son candidatos, siempre sujetos a KMO, Bartlett, tamaño muestral e interpretación de cargas.

In [ ]:
from src.pe3_visuals import kpi_cards
kpi_cards([
 ("Casos MCA", f"{len(d):,}", f"{len(d)/max(1,len(listings)):.1%} del universo completo"),
 ("Variables MCA activas", f"{len(active)}", "operación/fuente suplementarias"),
 ("Inercia MCA · 2 ejes", f"{100*inertia[:2].sum():.1f}%", "porción del total del espacio"),
 ("Barrios en análisis factorial", f"{n}", f"{p} indicadores urbanos"),
], title="MCA · composición y cobertura",
note="El muestreo, si se activa el tope computacional, es estratificado por operación.")
factor_ok = n>=5*p and kmo>=.60 and bart_p<.05 and nf>=1
kpi_cards([
 ("KMO", f"{kmo:.2f}", "≥0,60 como criterio orientativo"),
 ("Bartlett p", f"{bart_p:.3g}", "rechazo de matriz identidad"),
 ("Factores sobre umbral", str(nf), "parallel analysis, percentil 95"),
 ("Solución retenida", str(min(nf,3) if factor_ok else 0), "solo si se cumplen criterios mínimos"),
], title="Análisis factorial · adecuación",
note="La adecuación no sustituye estabilidad ni juicio sustantivo.")
top_modal = cats.nlargest(min(12,len(cats)),"masa").sort_values("masa")
fig, ax = plt.subplots(figsize=(8.5,4.8))
ax.barh(top_modal["modalidad"],100*top_modal["masa"],color="#4b78a8")
ax.set_xlabel("Masa de la modalidad (%)")
ax.set_title("MCA · modalidades con mayor peso en la muestra")
for spine in ("top","right"): ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8,4))
pos=np.arange(1,len(obs)+1)
ax.plot(pos,obs,marker="o",label="Observado",color="#4b78a8")
ax.plot(pos,threshold,marker="s",label="Aleatorio p95",color="#c45a46")
ax.set_xticks(pos); ax.set_xlabel("Componente"); ax.set_ylabel("Autovalor")
ax.set_title("Análisis factorial · parallel analysis")
ax.legend(frameon=False)
for spine in ("top","right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

**Lectura y límites.** Las tarjetas describen diagnóstico, no certifican un modelo. La masa de MCA ayuda a priorizar modalidades para leer, pero las categorías frecuentes no necesariamente son las más discriminantes. La parallel analysis no valida estabilidad espacial; revisar cargas y comunalidades antes de nombrar factores.